In [7]:
!pip install hdfs pyspark==3.5.1 --break-system-packages


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from hdfs import InsecureClient

# Nếu notebook ở host:
client = InsecureClient("http://namenode:9870", user="root")
client.status("/")

{'accessTime': 0,
 'blockSize': 0,
 'childrenNum': 9,
 'fileId': 16385,
 'group': 'supergroup',
 'length': 0,
 'modificationTime': 1773480423155,
 'owner': 'root',
 'pathSuffix': '',
 'permission': '755',
 'replication': 0,
 'snapshotEnabled': True,
 'storagePolicy': 0,
 'type': 'DIRECTORY'}

In [6]:

# Nếu notebook ở container:
# client = InsecureClient("http://namenode:9870", user="root")

# Upload file local -> HDFS
local_path = "./source_system/hadoop-3.4.0.tar.gz"
hdfs_path = "/tiktok/hadoop-3.4.0.tar.gz"

client.makedirs("/tiktok")
client.upload(hdfs_path, local_path, overwrite=True)
print(client.list("/tiktok"))


['hadoop-3.4.0.tar.gz']


In [7]:

# Nếu notebook ở container:
# client = InsecureClient("http://namenode:9870", user="root")

# Upload file local -> HDFS
local_path = "./source_system/run-wordcount.sh"
hdfs_path = "/tiktok/run-wordcount.sh"

client.makedirs("/tiktok")
client.upload(hdfs_path, local_path, overwrite=True)
print(client.list("/tiktok"))


['hadoop-3.4.0.tar.gz', 'run-wordcount.sh']


In [ ]:
# Download file to local then run.
client.download('/path/in/hdfs/file.txt', 'local_file.txt')

In [2]:
# Spark conf

In [11]:
from pyspark import SparkContext
from pyspark.sql import SparkSession

try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass

try:
    sc = SparkContext.getOrCreate()
    sc.stop()
except Exception:
    pass
SparkContext._active_spark_context = None

spark = SparkSession.builder \
    .appName("kafka-to-hdfs") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [16]:
data = [
    (1, "Alice", 29, "Hanoi"),
    (2, "Bob", 35, "HCM"),
    (3, "Cindy", 22, "Danang"),
    (4, "David", 41, "Hanoi"),
]*100000
df = spark.createDataFrame(data, ["id", "name", "age", "city"])
df.show()

+---+-----+---+------+
| id| name|age|  city|
+---+-----+---+------+
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
+---+-----+---+------+
only showing top 20 rows



In [17]:
output_path = "hdfs://namenode:8020/hadoop-data/output/mock_parquet"
df.write.mode("overwrite").parquet(output_path)

In [18]:
df2 = spark.read.parquet(output_path)
df2.show()

+---+-----+---+------+
| id| name|age|  city|
+---+-----+---+------+
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
|  1|Alice| 29| Hanoi|
|  2|  Bob| 35|   HCM|
|  3|Cindy| 22|Danang|
|  4|David| 41| Hanoi|
+---+-----+---+------+
only showing top 20 rows

